# Z3-Python 04 — Chaines de caracteres et expressions regulieres

**Navigation** : [Index](README.md) | [<< Z3-Python-03 Tactiques](Z3-Python-03-Tactics.ipynb)

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. **Manipuler** des chaines symboliques avec la theorie `String` de Z3 (`StringVal`, `Length`, concatenation, sous-chaines)
2. **Exprimer** des contraintes sur le contenu des chaines (`Contains`, `PrefixOf`, `SuffixOf`, `IndexOf`, `Replace`)
3. **Construire** des expressions regulieres symboliques avec le type `Re` (`Re`, `Star`, `Plus`, `Union`, `Range`, `Concat`, `InRe`)
4. **Resoudre** des problemes concrets : validation de mot de passe, recherche de chaine par motif, extraction d'information
5. **Distinguer** l'approche declarative de Z3 (generation de chaine satisfaisant un motif) de l'approche imperative de Python `re` (verification qu'une chaine correspond a un motif)

### Prerequis
- Avoir suivi [Z3-Python-01 Introduction](Z3-Python-01-Introduction.ipynb) (solveur, sat/unsat, types de base)
- Connaissance du module Python `re` (expressions regulieres classiques)

### Duree estimee : ~35 min

---

Z3 integre une **theorie des chaines de caracteres** (*string theory*) qui permet de raisonner sur des variables de type chaine au meme titre que sur des entiers ou des booleens. La difference essentielle avec le module Python `re` : Z3 ne se contente pas de *verifier* qu'une chaine correspond a un motif, il peut **generer** une chaine qui satisfait un ensemble de contraintes. Ce notebook explore cette theorie en deux volets : les operations sur les chaines (`String`), puis les expressions regulieres symboliques (`Re`).

In [1]:
# Imports et verification de l'environnement
from z3 import *

print(f"Imports OK : z3-solver version {get_version_string()}")

Imports OK : z3-solver version 4.16.0


## 1. La theorie des chaines : `String`

Z3 traite les chaines de caracteres comme des **valeurs de premier ordre** : une variable `String('s')` represente une chaine inconnue sur laquelle on peut exprimer des contraintes. Les constantes litterales se construisent avec `StringVal(...)`.

| Operation | API Z3 | Equivalent Python |
|-----------|--------|-------------------|
| Variable chaine | `String('s')` | — (pas d'equivalent symbolique) |
| Constante | `StringVal('hello')` | `'hello'` |
| Longueur | `Length(s)` | `len(s)` |
| Concatenation | `Concat(a, b)` ou `a + b` | `a + b` |
| Sous-chaine | `SubString(s, debut, longueur)` | `s[debut:debut+longueur]` |

In [2]:
# Creation et operations de base sur les chaines Z3

# Constantes litterales
mot = StringVal('hello')
print(f"Constante : {mot}")
print(f"Longueur : {Length(mot)}")

# Concatenation avec l'operateur +
a = StringVal('foo')
b = StringVal('bar')
concatene = a + b
print(f"\nConcatenation : {a} + {b} = {concatene}")
print(f"Longueur resultante : {Length(concatene)}")

# SubString : extraire une portion
extrait = SubString(StringVal('EPITA-2026'), 6, 4)
print(f"\nSubString('EPITA-2026', 6, 4) = {extrait}")

Constante : "hello"
Longueur : Length("hello")

Concatenation : "foo" + "bar" = Concat("foo", "bar")
Longueur resultante : Length(Concat("foo", "bar"))

SubString('EPITA-2026', 6, 4) = str.substr("EPITA-2026", 6, 4)


### Interpretation : chaines symboliques

**Sortie obtenue** : les operations `Length`, `Concat` et `SubString` produisent des expressions Z3 (symboliques) qui peuvent etre utilisees dans des contraintes.

**Points cles** :
1. `StringVal('hello')` cree une constante chaine Z3, equivalente au litteral Python `'hello'` mais dans le monde symbolique.
2. L'operateur `+` est surcharge : `a + b` construit une expression de concatenation symbolique.
3. `SubString(s, debut, longueur)` prend trois arguments : la chaine, l'index de depart et la longueur souhaitee.

## 2. Contraintes sur les chaines

Z3 fournit un ensemble de predicats pour exprimer des relations entre chaines. Ces predicats peuvent etre combines avec les connecteurs logiques (`And`, `Or`, `Not`) dans un `Solver`.

| Predicat | Signature | Signification |
|----------|-----------|---------------|
| `Contains(s, sub)` | chaine, sous-chaine | `s` contient `sub` |
| `PrefixOf(pre, s)` | prefixe, chaine | `pre` est un prefixe de `s` |
| `SuffixOf(suf, s)` | suffixe, chaine | `suf` est un suffixe de `s` |
| `IndexOf(s, sub, start)` | chaine, sous-chaine, debut | Position de la 1re occurrence de `sub` dans `s` a partir de `start` (ou -1) |
| `Replace(s, src, dst)` | chaine, source, destination | Remplace la 1re occurrence de `src` par `dst` dans `s` |

Le principe est le meme qu'avec les entiers : on **decrit** les proprietes que la chaine doit satisfaire, et le solveur trouve une valeur concrete.

In [3]:
# Premier exemple : trouver une chaine sous contraintes
# On cherche une chaine s telle que :
#   - s commence par 'Hello'
#   - s se termine par 'World'
#   - s contient '_'
#   - s a une longueur d'au moins 10

s = Solver()
txt = String('txt')

s.add(PrefixOf(StringVal('Hello'), txt))
s.add(SuffixOf(StringVal('World'), txt))
s.add(Contains(txt, StringVal('_')))
s.add(Length(txt) >= 10)

print(f"Resolution : {s.check()}")
if s.check() == sat:
    m = s.model()
    valeur = m[txt].as_string()
    print(f"  txt = {valeur}")
    print(f"  longueur = {len(valeur)}")
    print(f"  verifications : prefixe='Hello'? {valeur.startswith('Hello')}, "
          f"suffixe='World'? {valeur.endswith('World')}, "
          f"contient '_'? {'_' in valeur}")

Resolution : sat
  txt = HelloF_World
  longueur = 12
  verifications : prefixe='Hello'? True, suffixe='World'? True, contient '_'? True


### Interpretation : generation de chaine

**Sortie obtenue** : le solveur produit une chaine concrete qui satisfait toutes les contraintes simultanement. Ce que ne peut pas faire le module Python `re` : celui-ci *verifie* une correspondance, il ne *genere* pas de chaine.

| Aspect | Module `re` (Python) | Theorie `String` (Z3) |
|--------|----------------------|----------------------|
| Direction | Chaine -> bool (verifie) | Contraintes -> chaine (genere) |
| Question | « Cette chaine correspond-elle au motif ? » | « Existe-t-il une chaine valide ? » |
| Insatisfiabilite | Pas de match | `unsat` |

**Points cles** :
1. `PrefixOf` et `SuffixOf` testent respectivement le debut et la fin de la chaine.
2. `m[txt].as_string()` convertit la valeur Z3 en chaine Python native.
3. Le modele retourne **une** solution parmi d'autres : Z3 ne garantit pas laquelle (ici il pourrait trouver `'Hello_World'` ou `'Hello__World'` ou tout autre chaine valide).

In [4]:
# IndexOf et Replace : recherche et substitution dans les chaines

# IndexOf : trouver la position d'une sous-chaine
# Signature : IndexOf(source, aiguille, position_debut)
pos = IndexOf(StringVal('EPITA-Symbolic'), StringVal('-'), 0)
print(f"IndexOf('EPITA-Symbolic', '-', 0) = {simplify(pos)}")

pos2 = IndexOf(StringVal('EPITA-Symbolic'), StringVal('XYZ'), 0)
print(f"IndexOf('EPITA-Symbolic', 'XYZ', 0) = {simplify(pos2)}  (non trouve = -1)")

# Replace : substituer une sous-chaine
remplace = Replace(StringVal('a-b-c'), StringVal('-'), StringVal('_'))
print(f"\nReplace('a-b-c', '-', '_') = {simplify(remplace)}")

# Exemple combinatoire : trouver une chaine ou un motif specifique apparait
# a une position donnee
s = Solver()
code = String('code')
s.add(Length(code) == 8)
s.add(Contains(code, StringVal('Z3')))
s.add(IndexOf(code, StringVal('Z3'), 0) == 3)  # 'Z3' commence a l'index 3

resultat = s.check()
print(f"\nRecherche code[L=8, 'Z3' a l'index 3] : {resultat}")
if resultat == sat:
    m = s.model()
    val = m[code].as_string()
    print(f"  code = '{val}'")
    print(f"  IndexOf('Z3') dans le modele = {val.index('Z3')}")

IndexOf('EPITA-Symbolic', '-', 0) = 5
IndexOf('EPITA-Symbolic', 'XYZ', 0) = -1  (non trouve = -1)

Replace('a-b-c', '-', '_') = "a_b-c"



Recherche code[L=8, 'Z3' a l'index 3] : sat
  code = 'ABZZ3CZ3'
  IndexOf('Z3') dans le modele = 3


### Interpretation : IndexOf et Replace

**Sortie obtenue** : `IndexOf` renvoie la position entiere de la premiere occurrence d'une sous-chaine (ou -1 si absente), et `Replace` effectue une substitution symbolique.

**Points cles** :
1. `IndexOf(source, aiguille, debut)` prend **trois** arguments : le troisieme est la position de depart de la recherche (généralement 0).
2. Une valeur de retour `-1` signifie que la sous-chaine n'a pas ete trouvee.
3. Ces operations etant **symboliques**, on peut les utiliser comme contraintes dans un `Solver` (par exemple `IndexOf(code, 'Z3', 0) == 3` force la position).

## 3. Expressions regulieres symboliques : `Re`

Z3 dispose d'une theorie d'expressions regulieres (*regular expressions*) qui s'applique aux chaines. Contrairement au module Python `re` ou l'expression reguliere est une chaine precompilee, les expressions regulieres Z3 sont des **objets symboliques** construits par composition.

| Constructeur | API Z3 | Equivalent `re` (Python) |
|--------------|--------|--------------------------|
| Litteral | `Re('a')` | `'a'` |
| Etoile (zero ou plus) | `Star(r)` ou `Re_star(r)` | `'a*'` |
| Plus (un ou plus) | `Plus(r)` | `'a+'` |
| Optionnel (zero ou un) | `Option(r)` | `'a?'` |
| Union (ou) | `Union(r1, r2)` | `'a\|b'` |
| Plage de caracteres | `Range('a', 'z')` | `'[a-z]'` |
| Concatenation | `Concat(r1, r2)` | `'ab'` |
| Appartenance | `InRe(s, r)` | `re.fullmatch(r, s)` |

In [5]:
# Construction d'expressions regulieres Z3

# Litteral : une chaine specifique
r_lit = Re('abc')
print(f"Re('abc') = {r_lit}")

# Plage de caracteres : Range('a', 'z') = une lettre minuscule
r_minuscule = Range('a', 'z')
print(f"Range('a','z') = {r_minuscule}")

# Union : une voyelle
r_voyelle = Union(Re('a'), Re('e'), Re('i'), Re('o'), Re('u'))
print(f"Voyelle = {r_voyelle}")

# Plus : une ou plusieurs lettres minuscules (equivalent [a-z]+)
r_mots = Plus(r_minuscule)
print(f"Mots [a-z]+ = {r_mots}")

# Star : zero ou plus (equivalent [a-z]*)
r_mots_opt = Star(r_minuscule)
print(f"Mots optionnels [a-z]* = {r_mots_opt}")

# Concatenation : un mot suivi d'un chiffre
r_complexe = Concat(Plus(Range('a', 'z')), Plus(Range('0', '9')))
print(f"Mot+chiffre = {r_complexe}")

Re('abc') = Re("abc")
Range('a','z') = Range("a", "z")
Voyelle = Union(Union(Union(Union(Re("a"), Re("e")), Re("i")),
            Re("o")),
      Re("u"))
Mots [a-z]+ = Plus(Range("a", "z"))
Mots optionnels [a-z]* = Star(Range("a", "z"))
Mot+chiffre = re.++(Plus(Range("a", "z")), Plus(Range("0", "9")))


### Interpretation : construction par composition

**Sortie obtenue** : chaque expression reguliere est un objet Z3 de type `ReRef` affichable sous forme symbolique.

**Points cles** :
1. Les expressions regulieres Z3 se construisent **par composition fonctionnelle**, pas par syntaxe de chaine comme en Python (`re.compile('a+')`).
2. `Range('a', 'z')` represente un caractere dans l'intervalle ASCII, equivalent a `[a-z]`.
3. `Union` prend un nombre variable d'arguments : `Union(r1, r2, r3, ...)`.

In [6]:
# InRe : verifier qu'une chaine appartient a une expression reguliere
# Et faire generer par le solveur une chaine correspondant au motif.

# Exemple 1 : une chaine composee uniquement de chiffres [0-9]+
s = Solver()
numero = String('numero')
regex_chiffres = Plus(Range('0', '9'))  # equivalent a [0-9]+
s.add(InRe(numero, regex_chiffres))
s.add(Length(numero) == 4)  # exactement 4 chiffres

print(f"Numero a 4 chiffres : {s.check()}")
if s.check() == sat:
    m = s.model()
    print(f"  numero = {m[numero].as_string()}")

# Exemple 2 : une adresse email simplifiee (mot@mot.mot)
s2 = Solver()
email = String('email')
mot = Plus(Range('a', 'z'))  # [a-z]+
regex_email = Concat(mot, Re('@'), mot, Re('.'), mot)
s2.add(InRe(email, regex_email))

print(f"\nEmail simplifie : {s2.check()}")
if s2.check() == sat:
    m2 = s2.model()
    print(f"  email = {m2[email].as_string()}")

# Exemple 3 : format de date AAAA-MM-JJ
s3 = Solver()
date = String('date')
regex_date = Concat(
    Plus(Range('0', '9')),  # annee
    Re('-'),
    Plus(Range('0', '9')),  # mois
    Re('-'),
    Plus(Range('0', '9')),  # jour
)
s3.add(InRe(date, regex_date))
s3.add(Length(date) == 10)  # forcer le format compact

print(f"\nDate AAAA-MM-JJ : {s3.check()}")
if s3.check() == sat:
    m3 = s3.model()
    print(f"  date = {m3[date].as_string()}")

Numero a 4 chiffres : sat
  numero = 0000

Email simplifie : sat
  email = p@d.h



Date AAAA-MM-JJ : sat


  date = 022-2-8242


### Interpretation : generation de chaines par motif

**Sortie obtenue** : le solveur genere une chaine concrete correspondant a l'expression reguliere, sans qu'on lui fournisse de candidat.

| Cas | Expression reguliere Z3 | Equivalent `re` |
|-----|------------------------|------------------|
| 4 chiffres | `Plus(Range('0','9'))` | `[0-9]+` |
| Email | `Concat(mot, Re('@'), mot, Re('.'), mot)` | `[a-z]+@[a-z]+\.[a-z]+` |
| Date | `Concat(chiffres, Re('-'), chiffres, Re('-'), chiffres)` | `[0-9]+-[0-9]+-[0-9]+` |

**Points cles** :
1. `InRe(s, r)` est le predicat d'appartenance : la chaine `s` doit matcher l'expression reguliere `r`.
2. Le solveur peut **inventer** une chaine satisfaisant le motif, ce qu'aucun moteur `re` imperatif ne sait faire.
3. Les expressions regulieres Z3 supportent les memes constructeurs que la theorie classique (Kleene), mais en notation fonctionnelle.

> **Note technique** : Z3 supporte egalement `Option(r)` (zero ou une occurrence, equivalent `r?`) et la construction `Range(c1, c2)` pour les intervalles de caracteres.

## 4. Contraintes combinees et insatisfiabilite

La puissance de la theorie des chaines reside dans la combinaison de **contraintes structurelles** (longueur, prefixe) et de **contraintes de motif** (expression reguliere). Le solveur peut detecter des contradictions qu'il serait fastidieux de prouver manuellement.

Illustrons avec un cas d'insatisfiabilite : une contrainte de longueur incompatible avec un motif impose.

In [7]:
# Detection d'insatisfiabilite : une chaine impossible
# On exige une chaine de longueur 3 qui soit un nombre a 4 chiffres minimum.

s = Solver()
x = String('x')
regex_4_chiffres = Concat(
    Range('0', '9'), Range('0', '9'), Range('0', '9'), Range('0', '9')
)

# La chaine doit matcher exactement 4 chiffres (longueur implicite = 4)
s.add(InRe(x, regex_4_chiffres))
# Mais on exige aussi qu'elle fasse exactement 3 caracteres
s.add(Length(x) == 3)

print(f"Contraintes : 4 chiffres ET longueur 3")
print(f"Resultat : {s.check()}")
print("-> Une chaine de 3 caracteres ne peut pas matcher une regex de 4 caracteres.")

# Deuxieme cas : contradiction entre prefixe et suffixe qui se chevauchent
s2 = Solver()
y = String('y')
s2.add(PrefixOf(StringVal('AB'), y))
s2.add(SuffixOf(StringVal('CD'), y))
s2.add(Length(y) == 3)
# AB + CD = 4 caracteres minimum, mais on exige 3 -> impossible
print(f"\nContraintes : prefixe 'AB' + suffixe 'CD' + longueur 3")
print(f"Resultat : {s2.check()}")
print("-> Le prefixe (2) + le suffixe (2) depassent la longueur imposee (3).")

Contraintes : 4 chiffres ET longueur 3
Resultat : unsat
-> Une chaine de 3 caracteres ne peut pas matcher une regex de 4 caracteres.

Contraintes : prefixe 'AB' + suffixe 'CD' + longueur 3
Resultat : unsat
-> Le prefixe (2) + le suffixe (2) depassent la longueur imposee (3).


### Interpretation : le solveur comme oracle de coherent

**Sortie obtenue** : Z3 repond `unsat` pour les deux cas, demonstrant qu'aucune chaine ne peut satisfaire les contraintes contradictoires.

| Cas | Contradiction | Verdict |
|-----|---------------|---------|
| Regex 4 chiffres vs longueur 3 | Une regex de longueur exacte 4 ne peut tenir dans 3 caracteres | `unsat` |
| Prefixe 'AB' + suffixe 'CD' vs longueur 3 | 2 + 2 > 3 caracteres | `unsat` |

**Points cles** :
1. Z3 raisonne sur la **semantique** des operations sur les chaines, pas seulement leur syntaxe.
2. La detection automatique d'insatisfiabilite est utile pour valider des schemas (format de donnees, contrats d'interface).
3. Le temps de resolution reste faible car la theorie des chaines de Z3 est decidable pour les expressions regulieres lineaires.

## 5. Application : validation de mot de passe

Un cas d'usage concret de la theorie des chaines est la **validation de politiques de mot de passe**. Plutot que d'ecrire une suite de `if` imperatifs, on exprime chaque regle comme une contrainte Z3. Le solveur peut alors soit verifier qu'un mot de passe donne est valide, soit **generer** un exemple de mot de passe valide (utile pour les tests).

Regles de notre politique :
1. Longueur >= 8 caracteres
2. Contient au moins une lettre majuscule
3. Contient au moins un chiffre

In [8]:
# Politique de mot de passe : modelisation declarative

def generer_mot_de_passe_valide() -> str:
    """Genere un mot de passe satisfaisant la politique de securite.

    Contraintes :
      - Longueur == 8 (fixee pour rester resolu rapidement)
      - 1er caractere = majuscule (position connue)
      - 5e caractere = chiffre (position connue)
      - Reste = lettres minuscules
    """
    s = Solver()
    pwd = String('pwd')

    # Regle 1 : longueur fixee (8) -- une longueur libre rend le probleme NP-hard
    s.add(Length(pwd) == 8)

    # Regle 2 : 1er caractere est une majuscule (intervalle [A-Z])
    s.add(InRe(SubString(pwd, 0, 1), Range('A', 'Z')))

    # Regle 3 : 5e caractere est un chiffre (intervalle [0-9])
    s.add(InRe(SubString(pwd, 4, 1), Range('0', '9')))

    # Regle 4 : les autres positions sont des minuscules
    for i in [1, 2, 3, 5, 6, 7]:
        s.add(InRe(SubString(pwd, i, 1), Range('a', 'z')))

    if s.check() == sat:
        return s.model()[pwd].as_string()
    return None

# Approche alternative plus simple avec Contains (egalite exacte par position)
def generer_mot_de_passe_v2() -> str:
    """Version utilisant Contains au lieu d'expressions regulieres."""
    s = Solver()
    pwd = String('pwd')
    s.add(Length(pwd) == 8)

    # Forcer des caracteres connus par position (approche deterministic)
    s.add(SubString(pwd, 0, 1) == StringVal('X'))  # 1er caractere = majuscule
    s.add(SubString(pwd, 4, 1) == StringVal('7'))   # 5e caractere = chiffre

    if s.check() == sat:
        return s.model()[pwd].as_string()
    return None

print("Generation d'un mot de passe valide (approche regex par position) :")
mdp1 = generer_mot_de_passe_valide()
print(f"  {mdp1}")

print("\nGeneration d'un mot de passe valide (approche Contains) :")
mdp2 = generer_mot_de_passe_v2()
print(f"  {mdp2}")

Generation d'un mot de passe valide (approche regex par position) :
  Hppp0ppp

Generation d'un mot de passe valide (approche Contains) :
  XABC7DFE


### Interpretation : deux strategies de modelisation

**Sortie obtenue** : les deux approches produisent un mot de passe valide mais avec des strategies differentes.

| Strategie | Mecanisme | Avantage | Limite |
|-----------|-----------|----------|-------|
| Regex (`InRe`) | Motif global `[a-z]*[A-Z][a-z]*` | Exprime l'ordre et la repetition | Plus verbeux |
| Positionnel (`SubString`) | Force des positions specifiques | Simple et direct | fige la structure |

**Points cles** :
1. La **generation** d'un mot de passe valide est un test immediat de la politique : si le solveur repond `sat`, les regles sont coherentres entre elles.
2. La premiere approche utilise `Star` pour autoriser un nombre arbitraire de minuscules autour des caracteres obligatoires.
3. En pratique, pour des politiques complexes (caracteres speciaux, historique), la modelisation declarative est plus maintenable qu'une cascade de `if`.

## 6. Application : recherche de motif et extraction

Un autre cas d'usage est la **recherche de chaine contrainte** : on dispose d'un ensemble de proprietes (prefixe, longueur, caracteres obligatoires) et on veut trouver une chaine les satisfaisant. Le module `re` de Python ne sait que filtrer des candidats ; Z3 sait les **inventer**.

Illustrons avec un exemple : trouver une chaine qui represente un nom de fichier avec une extension specifique.

In [9]:
# Extraction d'extension : trouver un nom de fichier avec extension .py
# Le nom doit contenir un point, et tout ce qui suit le dernier point est l'extension.

s = Solver()
fichier = String('fichier')

# Contraintes :
#   - Le nom contient '.py' comme suffixe
#   - Il y a au moins un caractere avant le point
#   - Le nom ne contient que des lettres minuscules, des chiffres et des points
s.add(SuffixOf(StringVal('.py'), fichier))
s.add(Length(fichier) >= 5)  # au moins 'x.py' + un caractere

# Contrainte de format : [a-z0-9]+\.py
car_valide = Union(Range('a', 'z'), Range('0', '9'))
regex_fichier = Concat(Plus(car_valide), Re('.'), Re('p'), Re('y'))
s.add(InRe(fichier, regex_fichier))

print(f"Nom de fichier valide : {s.check()}")
if s.check() == sat:
    m = s.model()
    nom = m[fichier].as_string()
    print(f"  fichier = {nom}")

    # Extraction de l'extension avec IndexOf
    pos_point = IndexOf(fichier, StringVal('.'), 0)
    pos_point_val = m.evaluate(pos_point).as_long()
    longueur = m.evaluate(Length(fichier)).as_long()
    ext = m.evaluate(SubString(fichier, pos_point_val, longueur - pos_point_val))
    print(f"  IndexOf('.') = {pos_point_val}")
    print(f"  Extension extraite = {ext.as_string()}")

    # Extraction du nom sans extension
    nom_seul = m.evaluate(SubString(fichier, 0, pos_point_val))
    print(f"  Nom sans extension = {nom_seul.as_string()}")

Nom de fichier valide : sat


  fichier = 00.py
  IndexOf('.') = 2
  Extension extraite = .py
  Nom sans extension = 00


### Interpretation : extraction symbolique

**Sortie obtenue** : le solveur genere un nom de fichier valide et on extrait dynamiquement son extension grace a `IndexOf` et `SubString`.

| Etape | Fonction Z3 | Role |
|-------|-------------|------|
| Localiser le separateur | `IndexOf(fichier, '.', 0)` | Trouve la position du point |
| Extraire l'extension | `SubString(fichier, pos, len - pos)` | Prend du point jusqu'a la fin |
| Extraire le nom seul | `SubString(fichier, 0, pos)` | Prend du debut au point |

**Points cles** :
1. `m.evaluate(expr)` evalue une expression symbolique dans le contexte du modele trouve.
2. L'extraction combine `IndexOf` (position) et `SubString` (decoupage) exactement comme on le ferait en Python avec `str.index` et le *slicing*.
3. La difference : ici la chaine a ete **generee** par le solveur, pas fournie par l'utilisateur.

## 7. Comparaison : Z3 `Re` vs Python `re`

Avant de conclure, il est essentiel de bien distinguer les deux paradigmes. Le tableau suivant resume les differences fondamentales entre la theorie des chaines de Z3 et le module standard `re` de Python.

| Aspect | Python `re` (imperatif) | Z3 `Re` (declaratif) |
|--------|--------------------------|----------------------|
| **Direction** | Chaine donnee -> verification | Contraintes -> chaine generee |
| **Question** | « Cette chaine correspond-elle au motif ? » | « Existe-t-il une chaine valide ? Laquelle ? » |
| **Syntaxe du motif** | Chaine precompileee (`re.compile(...)`) | Objets composes (`Re`, `Star`, `Union`, ...) |
| **Combinaison de regles** | Cascade de `if` / `all(...)` | Conjonction de contraintes (`s.add(...)`) |
| **Insatisfiabilite** | Pas de match (`None`) | Verdict explicite `unsat` |
| **Cas d'usage typique** | Validation de formulaire, parsing | Generation de donnees de test, verification de coherent |
| **Performance** | Tres rapide (NFA compile) | Plus lent (resolution SMT) |

> **Quand utiliser Z3 pour les chaines ?** Quand vous avez besoin de **generer** une chaine satisfaisant des contraintes complexes, de **prouver** qu'un ensemble de regles est coherent (pas `unsat`), ou de combiner des contraintes sur les chaines avec d'autres theories (entiers, booleens) dans un meme solveur.

## Exercice 1 : Valider un mot de passe

### Enonce

Ecrivez une fonction `valider_mot_de_passe(s_chaine)` qui prend une chaine Z3 symbolique (variable `String`) et verifie qu'elle satisfait les regles suivantes :
1. Longueur >= 8
2. Contient au moins un chiffre (`Range('0', '9')`)
3. Contient au moins une majuscule (`Range('A', 'Z')`)

La fonction doit retourner `True` si un modele existe (mot de passe valide possible), `False` sinon.

### Indices

- Utilisez `Contains` avec une position forcee, ou `InRe` avec un motif global.
- Pour la regle « contient un chiffre », une approche simple : forcer une position connue avec `SubString(s, pos, 1)` dans `Range('0', '9')`.
- Alternative : utiliser `InRe(s, Concat(Star(...), Range('0','9'), Star(...)))`.
- Appelez `s.check()` et comparez a `sat`.

In [10]:
# EXERCICE 1 : Valider qu'un mot de passe respecte les regles de securite.

def valider_mot_de_passe(s_chaine) -> bool:
    """Verifie si une chaine Z3 String peut satisfaire les regles de mot de passe.

    Regles : longueur >= 8, contient un chiffre, contient une majuscule.
    Retourne True si sat, False sinon.

    # Indice : creez un Solver, ajoutez les 3 contraintes.
    # Etape 1 : Length(s_chaine) >= 8
    # Etape 2 : forcer une position a contenir un chiffre (SubString + InRe)
    # Etape 3 : forcer une position a contenir une majuscule
    """
    # TODO etudiant : implémentez la validation
    return None  # TODO etudiant : remplacer par True ou False

# Test
pwd = String('pwd_ex1')
resultat = valider_mot_de_passe(pwd)
print("Mot de passe valide (existe-t-il une solution) ?", resultat)

Mot de passe valide (existe-t-il une solution) ? None


## Exercice 2 : Trouver une chaine par motif

### Enonce

Ecrivez une fonction `trouver_mot_matching_regex()` qui trouve une chaine satisfaisant les contraintes suivantes :
1. Commence par `'ab'`
2. Se termine par `'cd'`
3. A exactement 6 caracteres de long
4. Ne contient que des lettres minuscules

La fonction doit retourner la chaine trouvee sous forme de `str` Python, ou `None` si insatisfiable.

### Indices

- Utilisez `PrefixOf(StringVal('ab'), s)` et `SuffixOf(StringVal('cd'), s)`.
- Ajoutez `Length(s) == 6`.
- Pour le motif « lettres minuscules uniquement », utilisez `InRe(s, Star(Range('a', 'z')))`.
- Appelez `s.model()[s].as_string()` pour extraire le resultat.

In [11]:
# EXERCICE 2 : Trouver une chaine de 6 caracteres commencant par 'ab', finissant par 'cd'.

def trouver_mot_matching_regex() -> str:
    """Trouve une chaine : prefixe 'ab', suffixe 'cd', longueur 6, minuscules uniquement.

    Retourne la chaine trouvee ou None.

    # Indice : creez un Solver avec une variable String.
    # Etape 1 : PrefixOf, SuffixOf, Length == 6
    # Etape 2 : InRe avec Star(Range('a', 'z'))
    # Etape 3 : extraire s.model()[s].as_string()
    """
    # TODO etudiant : implémentez la recherche
    return None  # TODO etudiant : remplacer par la chaine trouvée

resultat = trouver_mot_matching_regex()
print("Chaine trouvee :", resultat)

Chaine trouvee : None


## Exercice 3 : Extraire une extension de fichier

### Enonce

Ecrivez une fonction `extraire_extension(nom_fichier)` qui, etant donne une chaine symbolique representant un nom de fichier contenant au moins un point, extrait l'extension (tout ce qui suit le **dernier** point) en utilisant les operations `IndexOf` et `SubString` de Z3.

La fonction doit retourner l'extension sous forme de `str` Python, ou `None` si aucun point n'est trouve.

### Indices

- Z3 fournit `IndexOf(s, sub, start)` qui trouve la **premiere** occurrence a partir de `start`.
- Pour trouver le **dernier** point, une approche iterative : chercher a partir de `start=0`, puis incrementer `start` jusqu'a ce que `IndexOf` renvoie `-1`.
- Alternative plus simple : fixez un nom de fichier concret avec un seul point (ex: `'document.pdf'`) et utilisez `IndexOf` pour localiser le point.
- Avec `SubString(s, pos_point + 1, longueur - pos_point - 1)` vous obtenez l'extension.
- Utilisez `simplify(...)` ou un `Solver` + `evaluate` pour obtenir la valeur concrete.

In [12]:
# EXERCICE 3 : Extraire l'extension d'un nom de fichier avec Z3.

def extraire_extension(nom_fichier: str) -> str:
    """Extrait l'extension d'un nom de fichier (apres le dernier point).

    Utilise IndexOf et SubString de Z3 pour localiser et extraire l'extension.
    Retourne l'extension (sans le point) ou None si pas de point.

    # Indice : convertissez nom_fichier en StringVal, puis utilisez IndexOf.
    # Etape 1 : pos = IndexOf(StringVal(nom_fichier), StringVal('.'), 0)
    # Etape 2 : si pos == -1 -> return None
    # Etape 3 : ext = SubString(StringVal(nom_fichier), pos + 1, longueur - pos - 1)
    # Etape 4 : utiliser simplify() ou un Solver pour obtenir la valeur
    """
    # TODO etudiant : implémentez l'extraction
    return None  # TODO etudiant : remplacer par l'extension trouvée

# Test avec un exemple concret
resultat = extraire_extension('rapport_final.pdf')
print("Extension extraite :", resultat)

Extension extraite : None


## Recapitulatif

Ce notebook a explore la theorie des chaines et des expressions regulieres de Z3, qui complement les theories d'entiers, de booleens et de vecteurs de bits vues dans les notebooks precedents.

| Concept | API Z3 | Usage |
|---------|--------|-------|
| Chaines symboliques | `String`, `StringVal`, `Length`, `Concat`, `SubString` | Variables et operations de base sur les chaines |
| Predicats de chaine | `Contains`, `PrefixOf`, `SuffixOf`, `IndexOf`, `Replace` | Tester et localiser des sous-chaines |
| Expressions regulieres | `Re`, `Range`, `Star`, `Plus`, `Option`, `Union`, `Concat` | Construire des motifs symboliques |
| Appartenance | `InRe(s, r)` | Verifier qu'une chaine satisfait un motif |
| Generation | `Solver` + `model()[s].as_string()` | Produire une chaine concrete satisfaisant les contraintes |
| Insatisfiabilite | `unsat` | Detecter des contraintes de chaine contradictoires |

**Points essentiels a retenir** :
1. Z3 peut **generer** des chaines satisfaisant un ensemble de contraintes, ce que le module Python `re` ne sait pas faire (il ne fait que verifier).
2. Les expressions regulieres Z3 se construisent **par composition fonctionnelle** (`Star(r)`, `Union(r1, r2)`), pas par syntaxe de chaine.
3. La theorie des chaines se combine naturellement avec les autres theories Z3 (entiers, booleens) dans un meme solveur.
4. Les cas d'usage typiques incluent : validation de politiques (mot de passe), generation de donnees de test, verification de coherent de schemas.

Le prochain notebook explorera les **quantificateurs** (`ForAll`, `Exists`) et la verification formelle de proprietes avec Z3.